# Episode 7 — Modulation: Signals Controlling Parameters

⏱ About 30 minutes · Difficulty ★★★

**Musical element this episode controls: movement** — the filter opening and closing, the rhythm of a wobble, pitch glides.

**The equivalent Max patch:** the outputs of `adsr~`, `cycle~ 3` and `line~` patched into the cutoff inlet of `lores~` and the frequency inlet of `phasor~`.

**What you will hear:**
1. a filter sweep: the sound gradually opening up
2. a filter envelope: a pluck bass with a "pew" character
3. LFO → cutoff: "wub wub wub", locked to a BPM
4. tremolo, vibrato and a pitch drop
5. a complete wobble bass

**By the end you can:**
- patch any array into cutoff, volume or wavetable position
- explain why modulation amount is measured in octaves
- lock an LFO rate to a BPM
- explain why `freq * t` is wrong when pitch changes, and what `phasor~` really does

## Toolbox

Everything built in earlier episodes, copied as-is. Just run it; no need to re-read (the tricky lines carry `#` comments).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

sr = 44100

def play(x, gain=0.3):
    # np.clip holds anything beyond ±1 so nothing blows up
    return Audio(np.clip(gain * np.asarray(x), -1, 1), rate=sr, normalize=False)

def scope(x, title="", seconds=0.03):
    n = int(sr * seconds)                      # only plot this many samples from the start
    plt.figure(figsize=(9, 2.5))
    plt.plot(np.arange(n) / sr, np.asarray(x)[:n])
    plt.xlabel("time (s)")
    plt.title(title)
    plt.show()

def make_table(harmonic_amps, N=2048):
    p = np.arange(N) / N                       # phase of every table position, 0 to 1
    table = np.zeros(N)                        # start from an empty table of zeros
    for k, amp in enumerate(harmonic_amps, start=1):
        # harmonic k: k cycles across the table, at level amp
        table += amp * np.sin(2 * np.pi * k * p)
    return table / np.max(np.abs(table))       # normalize: peak becomes 1

def make_frames(F=16, max_harmonics=32, N=2048):
    frames = []
    for f in range(F):
        # how many harmonics frame f gets: rising evenly from 1 to max_harmonics
        n_h = 1 + round(f * (max_harmonics - 1) / (F - 1))
        frames.append(make_table([1 / k for k in range(1, n_h + 1)], N))
    return np.array(frames)                    # stack into a 2-D array: F rows × N columns

def osc_wavetable(frames, phase, pos):
    F, N = frames.shape

    # --- which two frames are we between ---
    fpos = np.clip(pos, 0, 1) * (F - 1)        # turn 0..1 into a frame number 0..F-1, decimals allowed
    f0 = np.floor(fpos).astype(int)            # the frame below
    f1 = np.minimum(f0 + 1, F - 1)             # the frame above (never past the last one)
    ffrac = fpos - f0                          # how far above the lower frame, 0..1

    # --- which two positions inside a table (same as Episode 3) ---
    p = phase * N
    i0 = np.floor(p).astype(int) % N           # left position
    i1 = (i0 + 1) % N                          # right position, wrapping to 0 at the end
    frac = p - np.floor(p)                     # the fractional part

    # interpolate inside each frame first, then between the two frames
    a = (1 - frac) * frames[f0, i0] + frac * frames[f0, i1]
    b = (1 - frac) * frames[f1, i0] + frac * frames[f1, i1]
    return (1 - ffrac) * a + ffrac * b

def adsr(t, a, d, s, r, gate):
    a, d, r = max(a, 1e-4), max(d, 1e-4), max(r, 1e-4)    # times must not be 0, or we would divide by zero below
    # while held: (0,0) → (a,1) → (a+d,s), then stays at s
    held = np.interp(t, [0, a, a + d], [0, 1, s])
    # how high the envelope is at the moment of release (attack may not be finished)
    level_at_release = np.interp(gate, [0, a, a + d], [0, 1, s])
    # fall from that level to 0 over r seconds; np.clip stops it going negative
    released = level_at_release * np.clip(1 - (t - gate) / r, 0, 1)
    return np.where(t < gate, held, released)  # held before release, released after

def lowpass(x, cutoff, res=0.0):
    x = np.asarray(x, dtype=float)
    n_samples = len(x)
    # cutoff may be one number or an array: make it an array as long as the sound
    cutoff = np.clip(np.broadcast_to(cutoff, n_samples), 20, sr * 0.45)

    # --- turn cutoff / res into the filter's internal coefficients (one set per sample) ---
    g = np.tan(np.pi * cutoff / sr)
    k = 1.414 * (1 - res) + 0.05 * res         # damping: higher res → smaller k → stronger resonance
    a1 = 1 / (1 + g * (g + k))
    a2 = g * a1
    a3 = g * a2

    # .tolist() turns arrays into plain lists: much faster to read one by one in a loop
    xs, A1, A2, A3 = x.tolist(), a1.tolist(), a2.tolist(), a3.tolist()
    y = [0.0] * n_samples
    ic1 = ic2 = 0.0                            # two state variables = two history objects in gen~

    for n in range(n_samples):                 # one sample at a time, in order
        v3 = xs[n] - ic2
        v1 = A1[n] * ic1 + A2[n] * v3
        v2 = ic2 + A2[n] * ic1 + A3[n] * v3    # v2 is the lowpass output
        ic1 = 2 * v1 - ic1                     # update the state for the next sample
        ic2 = 2 * v2 - ic2
        y[n] = v2
    return np.array(y)

frames = make_frames()

freq = 55
dur = 2.0
n_samples = int(sr * dur)
t = np.arange(n_samples) / sr
phase = (freq * t) % 1

saw = osc_wavetable(frames, phase, 1.0)

`saw` is a low 55 Hz (A1) saw lasting 2 seconds. Most examples in this episode use it as raw material.

## 1. Recall: a number vs an array

The conclusion of Episode 4:

> Pass one number and it is static, pass an array and it moves.

The `cutoff` of `lowpass(x, cutoff, res)` behaves the same. In Max this is the difference between patching a number and patching a signal into the cutoff inlet.

**All of modulation comes down to this: build an array and patch it into a parameter.** The rest of this episode merely swaps in different arrays and different parameters.

## 2. Filter sweep (`line~` → cutoff)

Start with `np.linspace` from Episode 4 and take the cutoff in a straight line from 100 Hz to 6000 Hz.

**You should hear:** the sound brightening quickly at first, then barely changing for most of the remaining time.

In [ ]:
cutoff_linear = np.linspace(100, 6000, n_samples)
play(lowpass(saw, cutoff_linear, res=0.5))

Why is it uneven? Because the ear hears frequency by **ratio** (octaves), not by Hz.

- 100 → 200 Hz is one octave
- 3000 → 6000 Hz is also one octave

On a straight line the first octave covers only 100 Hz (gone in an instant) while the last octave covers 3000 Hz (the entire second half).

`np.geomspace(100, 6000, n)`: also from 100 to 6000, but **each step multiplies by the same number** rather than adding the same number. Every octave then takes equally long.

**You should hear:** dull to bright, opening at a steady rate throughout.

In [ ]:
cutoff_geo = np.geomspace(100, 6000, n_samples)

plt.figure(figsize=(9, 3))
plt.plot(t, cutoff_linear, label="linspace")
plt.plot(t, cutoff_geo, label="geomspace")
plt.xlabel("time (s)")
plt.ylabel("cutoff (Hz)")
plt.legend()
plt.show()

play(lowpass(saw, cutoff_geo, res=0.5))

**Remember:** anything to do with frequency (cutoff, pitch) moves by ratio.

## 3. Filter envelope (`adsr~` → cutoff)

Build an envelope with `adsr()` from Episode 5 and let it push the cutoff.

The envelope runs from 0 to 1. How do we turn that into Hz? By ratio, as concluded above:

`cutoff = base * 2 ** (amount * env)`

- `2 ** octaves` = go up that many octaves (`2 ** 1` = ×2, `2 ** 2` = ×4, `2 ** 3` = ×8)
- at `env = 0`: `2 ** 0 = 1`, so cutoff equals `base`
- at `env = 1`: cutoff is `amount` octaves above `base`

So `base` is the filter's cutoff knob and `amount` is the envelope amount knob, measured in octaves.

In [ ]:
flt_env = adsr(t, a=0.001, d=0.25, s=0.0, r=0.1, gate=1.0)
amp_env = adsr(t, a=0.005, d=0.6, s=0.0, r=0.1, gate=1.0)

base = 150
amount = 5
cutoff = base * 2 ** (amount * flt_env)        # env 0 → 150 Hz; env 1 → 150 × 2^5 = 4800 Hz

plt.figure(figsize=(9, 2.5))
plt.plot(t, cutoff)
plt.xlabel("time (s)")
plt.ylabel("cutoff (Hz)")
plt.show()

play(lowpass(saw, cutoff, res=0.5) * amp_env)

**You should hear:** a pluck bass that starts bright and quickly turns dull, with a bit of a "pew".

Note the **two** envelopes: `flt_env` handles the cutoff, `amp_env` the loudness. The filter envelope is shorter than the amp envelope, so the note goes dull first and quiet afterwards.

### The size of the amount

**You should hear:** three plucks. With amount 1 there is hardly any "pew"; with 3 it is clear; with 5 it is exaggerated.

In [ ]:
for amount in [1, 3, 5]:
    print("amount =", amount, "octaves")
    display(play(lowpass(saw, 150 * 2 ** (amount * flt_env), res=0.5) * amp_env))

## 4. LFO → cutoff (`cycle~ 3` → cutoff)

An LFO is just a very slow oscillator: Episode 1's sine with the frequency set to 3 Hz.

`lfo` swings between -1 and 1. Put it into the same formula:

`cutoff = 300 * 2 ** (2 * lfo)`

- `lfo = -1` → 2 octaves below 300 = 75 Hz
- `lfo = 0` → 300 Hz
- `lfo = +1` → 2 octaves above 300 = 1200 Hz

**You should hear:** "wub wub wub", three per second.

In [ ]:
lfo_rate = 3
lfo = np.sin(2 * np.pi * lfo_rate * t)         # the same sine as Episode 1, just very slow

cutoff = 300 * 2 ** (2 * lfo)

plt.figure(figsize=(9, 2.5))
plt.plot(t, cutoff)
plt.xlabel("time (s)")
plt.ylabel("cutoff (Hz)")
plt.show()

play(lowpass(saw, cutoff, res=0.6))

### Locking to a BPM

LFO rate in Hz = times per second. Musically you would rather say "times per beat".

- `bpm / 60` = beats per second
- times "per beat" = times per second = the LFO's Hz

At 140 BPM with 2 per beat (eighth notes): `140 / 60 * 2 = 4.67 Hz`.

**You should hear:** a wobble at eighth notes of 140 BPM. About 9 of them in 2 seconds.

In [ ]:
bpm = 140
per_beat = 2                                   # 2 per beat = eighth notes; 4 gives sixteenths, 3 gives triplets
lfo_rate = bpm / 60 * per_beat
print("LFO rate:", round(lfo_rate, 2), "Hz")

lfo = np.sin(2 * np.pi * lfo_rate * t)
play(lowpass(saw, 300 * 2 ** (2 * lfo), res=0.6))

### LFO shapes

An LFO is an oscillator, so everything from Episode 2 applies: build a slow phase, then turn it into different shapes.

- **sine**: opens and closes smoothly → "wub"
- **saw down** (falling from 1 to -1): snaps open, then closes slowly → "yoi yoi"
- **square**: only two states, open and closed → bright and dark alternating

**You should hear:** three wobbles of different character at the same speed.

In [ ]:
lfo_phase = (lfo_rate * t) % 1                 # a slow phasor~

lfo_shapes = {
    "sine":     np.sin(2 * np.pi * lfo_phase),
    "saw down": 1 - 2 * lfo_phase,                         # 1 → -1
    "square":   np.where(lfo_phase < 0.5, 1.0, -1.0),
}

fig, axes = plt.subplots(3, 1, figsize=(9, 4), sharex=True)
for ax, name in zip(axes, lfo_shapes):
    ax.plot(t, lfo_shapes[name])
    ax.set_ylabel(name)
axes[-1].set_xlabel("time (s)")
plt.show()

for name in lfo_shapes:
    print(name)
    display(play(lowpass(saw, 300 * 2 ** (2 * lfo_shapes[name]), res=0.6)))

## 5. The same LFO, other parameters

Modulation is not limited to cutoff.

### Volume → tremolo

Volume cannot be negative, so shift the -1..1 LFO to 0..1: `0.5 + 0.5 * lfo`.

**You should hear:** the volume pulsing at eighth notes of 140 BPM.

In [ ]:
tremolo = 0.5 + 0.5 * lfo                      # -1..1 → 0..1
play(lowpass(saw, 800) * tremolo)

### Wavetable position

As in Episode 4. **You should hear:** the timbre itself moving between dark and bright; similar to a filter wobble but "cleaner", since there is no resonance.

In [ ]:
play(osc_wavetable(frames, phase, 0.5 + 0.5 * lfo))

## 6. Pitch modulation: there is a trap here

Say the pitch should glide from 110 Hz to 220 Hz over 2 seconds (one octave up). The intuitive move is to replace `freq` with an array:

In [ ]:
freq_glide = np.geomspace(110, 220, n_samples)

wrong_phase = (freq_glide * t) % 1             # looks reasonable, but is wrong
play(np.sin(2 * np.pi * wrong_phase))

**You should hear:** the pitch rising **clearly more** than one octave (it ends near 370 Hz rather than 220 Hz). That is wrong.

**Why:** `freq * t` means "the frequency has been this value **all along**, so this many cycles have elapsed". Once the frequency changes, that is false. At second 2 it computes "220 × 2 = 440 cycles", as if the frequency had been 220 Hz from the start — but for most of that time it was lower, and nowhere near that many cycles fit. To make up the number, phase is forced to run faster and faster.

### What `phasor~` really does

It does not look at "from the start until now", only at **how far to advance during this one sample**, and it keeps a running total.

- `freq` cycles per second, `sr` samples per second
- so **each sample advances by `freq / sr` cycles**
- add up every step and you have the phase

The "running total" is `np.cumsum` (cumulative sum): the n-th number is the sum of the first n numbers.

In [ ]:
steps = np.array([1, 2, 3, 4])
print("steps  =", steps)
print("cumsum =", np.cumsum(steps))            # 1, 1+2, 1+2+3, 1+2+3+4

This too is a way of "remembering the last time" — the same idea as last episode's `history`. Running totals are so common that numpy provides `cumsum`, so no hand-written loop is needed.

This is called **phase accumulation**.

In [ ]:
def phasor(freq, n_samples):
    # freq / sr = how far phase advances during one sample
    # cumsum adds the steps up; % 1 keeps the fractional part
    return np.cumsum(np.broadcast_to(freq, n_samples) / sr) % 1

right_phase = phasor(freq_glide, n_samples)
play(np.sin(2 * np.pi * right_phase))

**You should hear:** exactly one octave up, landing on 220 Hz.

What `np.broadcast_to(freq, n_samples)` does: if `freq` is one number it becomes `n_samples` copies of it; if it already is an array it is used as is. So both `phasor(110, n)` and `phasor(some_array, n)` work.

**From now on phase is always made with `phasor(freq, n_samples)`.** The earlier `(freq * t) % 1` is only right when freq never changes.

### Vibrato

Pitch moves by ratio too. A semitone is 1/12 of an octave, so:

`freq = 220 * 2 ** (semitones / 12 * lfo)`

**You should hear:** a note whose pitch wavers 5 times per second, half a semitone up and down, like a singer's vibrato.

In [ ]:
vib_lfo = np.sin(2 * np.pi * 5 * t)
freq_vib = 220 * 2 ** (0.5 / 12 * vib_lfo)     # 0.5 semitones up and down

play(osc_wavetable(frames, phasor(freq_vib, n_samples), 0.8))

### Modulating a modulator

A singer rarely wavers from the first instant; the note starts straight and vibrato is added gradually.

To do that, let the **depth** of the vibrato change over time too — multiply the LFO by an envelope. Depth is 0 for the first 0.5 seconds, then rises to 1 over 1 second.

**You should hear:** a steady note at first, then vibrato gradually appearing and deepening.

In [ ]:
depth = np.interp(t, [0, 0.5, 1.5], [0, 0, 1])             # 0 until 0.5 s, reaching 1 at 1.5 s
freq_vib = 220 * 2 ** (0.7 / 12 * vib_lfo * depth)         # LFO × depth

play(osc_wavetable(frames, phasor(freq_vib, n_samples), 0.8))

### Pitch drop

The pitch falls quickly from high (4 octaves above 50 Hz = 800 Hz) to 50 Hz, with a short amp envelope.

**You should hear:** a "boom / pew", like an electronic kick or a laser. A shorter `d=0.12` leans toward a kick, a longer one toward a laser.

In [ ]:
pitch_env = adsr(t, a=0.001, d=0.12, s=0.0, r=0.1, gate=1.0)
freq_drop = 50 * 2 ** (4 * pitch_env)                      # env 1 → 800 Hz; env 0 → 50 Hz
kick_amp = adsr(t, a=0.001, d=0.5, s=0.0, r=0.1, gate=1.0)

play(np.sin(2 * np.pi * phasor(freq_drop, n_samples)) * kick_amp, gain=0.6)

## 7. Everything patched together: a wobble bass

- oscillator: a 55 Hz wavetable
- one saw-down LFO (eighth notes at 140 BPM) pushing both the **wavetable position** and the **cutoff**
- a touch of the LFO on pitch as well, to make it "dirtier"

**You should hear:** a typical "yoi yoi yoi" wobble bass.

In [ ]:
wob = lfo_shapes["saw down"]                                   # -1..1
wob01 = 0.5 + 0.5 * wob                                        # 0..1

bass_phase = phasor(55 * 2 ** (0.3 / 12 * wob), n_samples)     # pitch: 0.3 semitones up and down
bass = osc_wavetable(frames, bass_phase, 0.2 + 0.8 * wob01)    # wavetable position: 0.2..1
bass = lowpass(bass, 200 * 2 ** (3 * wob01), res=0.7)          # cutoff: 200..1600 Hz

plt.figure(figsize=(9, 3.5))
plt.specgram(bass + 1e-9, NFFT=2048, Fs=sr, noverlap=1792, cmap="magma", vmin=-120)
plt.ylim(0, 3000)
plt.xlabel("time (s)")
plt.ylabel("frequency (Hz)")
plt.show()

play(bass, gain=0.2)

Each "bulge" in the spectrogram is one "yoi": the bright region shoots up and slowly settles back — exactly the shape of the saw-down LFO.

## Common mistakes

| Symptom | Cause |
|---|---|
| A sweep sounds uneven | `linspace` was used for a frequency; use `geomspace` or `base * 2 ** (...)` |
| A pitch glide covers the wrong range | `(freq * t) % 1` was used. When freq changes you must use `phasor()` |
| Cutoff goes negative / errors | `base + amount * lfo` (addition) was used. The multiplying form never goes negative |
| Tremolo sounds like distortion | The volume LFO was not shifted to 0..1; its negative half flips the waveform |

## Exercises

**1.** Change the BPM wobble of section 4 to sixteenth notes (4 per beat), then to triplets (3 per beat).

<details><summary>Answer</summary>

```python
for per_beat in [4, 3]:
    rate = 140 / 60 * per_beat
    l = np.sin(2 * np.pi * rate * t)
    display(play(lowpass(saw, 300 * 2 ** (2 * l), res=0.6)))
```
</details>

**2.** Make a note that glides **up**: from 110 Hz to 440 Hz in 0.3 seconds, then holds. Hint: build a 0 → 1 line with `np.interp`, then use `110 * 2 ** (2 * line)`.

<details><summary>Answer</summary>

```python
rise = np.interp(t, [0, 0.3], [0, 1])
play(np.sin(2 * np.pi * phasor(110 * 2 ** (2 * rise), n_samples)))
```
</details>

**3.** Make the LFO **rate** itself speed up, from 1 Hz to 12 Hz. Hint: an LFO is an oscillator, so when its rate changes its phase needs `phasor()` too.

<details><summary>Answer</summary>

```python
rate = np.geomspace(1, 12, n_samples)
l = np.sin(2 * np.pi * phasor(rate, n_samples))
play(lowpass(saw, 300 * 2 ** (2 * l), res=0.6))
```
The wobble gets faster and faster.
</details>

## Recap

| Max | Python |
|---|---|
| a signal into the cutoff inlet | `lowpass(x, cutoff_array, res)` |
| `line~` (linear) | `np.linspace(a, b, n)` |
| `line~` → `mtof~` (moving in pitch) | `np.geomspace(a, b, n)` |
| LFO: `cycle~ 3` | `np.sin(2 * np.pi * 3 * t)` |
| mod amount (octaves) | `base * 2 ** (amount * mod)` |
| `phasor~` (signal-rate frequency) | `phasor(freq, n)` = `np.cumsum(freq / sr) % 1` |
| `+=` (gen~) / `accum` | `np.cumsum(...)` |

- modulation = build an array, patch it into a parameter
- frequency-like parameters move by ratio: `base * 2 ** (octaves)`
- when pitch changes, phase must be computed by accumulation

**Next episode:** oscillator, filter, envelopes and modulation packed into one function — a complete synth.